In [ ]:
# Core numerics and tensor ops
import torch
import numpy as np

# IO and image utilities
import cv2
import os
import random as rand
import torchvision
import pandas as pd

# Progress and plotting
from tqdm import tqdm
from torch import nn, Tensor
import matplotlib.pyplot as plt
from typing import Optional
from torch.nn import functional as F

# Data transforms and batching
from torchvision.transforms import v2 as T
from torchvision.utils import make_grid
from torch.utils.data import Dataset, DataLoader
from math import ceil

# GAN evaluation metrics
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

## GAN Mathematical Roadmap (Mapped to This Notebook)

This notebook implements a minimax GAN where:

$$
z \sim p_z(z),\quad \hat{x} = G_\theta(z),\quad D_\phi(x) \in \mathbb{R}
$$

- `Generator` learns a mapping from latent space to image space.
- `Discriminator` outputs a **logit** (not probability), trained with `BCEWithLogitsLoss`.

Core objective (original GAN form):

$$
\min_{\theta}\max_{\phi}\; \mathbb{E}_{x\sim p_{\text{data}}}[\log \sigma(D_\phi(x))] + \mathbb{E}_{z\sim p_z}[\log(1-\sigma(D_\phi(G_\theta(z))))]
$$

In code, this is implemented as two alternating supervised losses:

- Discriminator step: classify real as 1 (smoothed to 0.9) and fake as 0.
- Generator step: push fake logits toward label 1.

This decomposition is numerically stable because logits are fed directly into `BCEWithLogitsLoss`.

### Code-to-math map for this notebook

- Imports cell: defines the operators used in $G_\theta$, $D_\phi$, and the metrics.
- Dataset cell: constructs $x \in [-1,1]^{3\times 32\times 32}$, matching generator output range.
- Blocks/model cell: parameterizes $G_\theta$ and $D_\phi$ and implements $\mathcal{L}_D, \mathcal{L}_G$.
- Training cell: performs stochastic optimization updates for $\theta$ and $\phi$.
- Metrics cell: estimates distance/diversity proxies (FID, IS) from samples.

In [ ]:
class CIFAR(Dataset):
    def __init__(self, path="/home/chaitanya-kohli/GAN-Him/tiny-imagenet/train/n07920052", dataset: Optional[list] = None):
        super().__init__()
        self.path = path
        self.files = os.listdir(self.path) if dataset is None else dataset
        
        # x in [0, 1] -> 2x - 1 in [-1, 1], matching tanh generator outputs.
        self.T = T.Compose([
            T.ToImage(),
            T.ToDtype(torch.float32, scale=True),
            T.Resize((32, 32)),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        file = self.files[idx]
        img_path = os.path.join(self.path, file)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.T(img)
        return img


class MNISTCSV(Dataset):
    def __init__(self, csv_file):
        data = pd.read_csv(csv_file)
        
        self.labels = torch.tensor(data.iloc[:, 0].values, dtype=torch.long)
        images = data.iloc[:, 1:].values.reshape(-1, 28, 28).astype("uint8")
        self.images = torch.from_numpy(images)

        # Promote grayscale -> RGB-like 3 channels to reuse same discriminator architecture.
        self.transforms = T.Compose([
            T.ToImage(),
            T.Grayscale(num_output_channels=3),
            T.Resize((32, 32)),
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        
        if self.transforms:
            img = self.transforms(img)
            
        return img, label

## Data Pipeline Math: How Inputs Are Scaled

Both datasets apply transforms that place images in a normalized range suitable for `tanh` generator outputs.

For input pixel values in $[0,1]$, the normalization in code is:

$$
x_{\text{norm}} = \frac{x-0.5}{0.5} = 2x-1 \in [-1,1]
$$

This matches the generator output activation:

$$
\hat{x} = \tanh(a) \in [-1,1]
$$

So real and generated samples live in the same value domain before being sent to the discriminator.

### Dataset-specific mapping

- `CIFAR` class: reads RGB with OpenCV, resizes to $32\times 32$, normalizes to $[-1,1]$.
- `MNISTCSV` class: reshapes flat vectors to $28\times 28$, converts to 3 channels, resizes to $32\times 32$, normalizes similarly.

Mathematically, each mini-batch becomes a tensor:

$$
X \in \mathbb{R}^{B\times 3\times 32\times 32}
$$

which is the shared input/output shape for discriminator/generator training.

In [ ]:
class GenBlock(nn.Module):
    def __init__(self, in_channel, out_channel, is_final):
        super().__init__()
        # Conv-BN-activation stack approximates one stage of learned upsampling refinement.
        layers = [
            nn.Conv2d(in_channel, (out_channel + in_channel) // 2, 3, 1, 1),
            nn.BatchNorm2d((out_channel + in_channel) // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d((in_channel + out_channel) // 2, out_channel, 3, 1, 1),
        ]
        
        if not is_final:
            layers.append(nn.BatchNorm2d(out_channel))
            layers.append(nn.LeakyReLU(0.2, inplace=True))

        # Spatial upsampling x2 after feature refinement.
        layers.append(nn.UpsamplingNearest2d(scale_factor=2))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class DisBlock(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(in_channel, (out_channel + in_channel) // 2, 3, 1, 1),
            nn.BatchNorm2d((out_channel + in_channel) // 2),
            nn.LeakyReLU(0.2),
            nn.Conv2d((out_channel + in_channel) // 2, out_channel, 3, 1, 1),
            nn.BatchNorm2d(out_channel),
            nn.LeakyReLU(0.2),
            # Downsample x2 to aggregate context and increase receptive field.
            nn.MaxPool2d(2, 2),
        )
    
    def forward(self, x):
        return self.layer(x)


class ResGenBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
        )
        
        # Shortcut path enforces residual form: y = F(x) + S(x).
        self.shortcut = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.conv_block(x) + self.shortcut(x)

## Generator/Discriminator Blocks as Functions

### Generator-side blocks (`GenBlock`, `ResGenBlock`)

Each upsampling stage approximates a learned refinement:

$$
h^{(l+1)} = \text{Up}\left(\varphi\left(\text{BN}(W_2 * \varphi(\text{BN}(W_1 * h^{(l)})))\right)\right)
$$

where $*$ is convolution and $\varphi$ is `LeakyReLU` or `ReLU` depending on block type.

For residual blocks:

$$
h^{(l+1)} = F(h^{(l)}) + S(h^{(l)})
$$

with $F$ the 2-conv path and $S$ the shortcut path. This improves gradient flow and stabilizes deeper generators.

### Discriminator-side blocks (`DisBlock`)

Each block performs feature extraction + downsampling:

$$
h^{(l+1)} = \text{Pool}\left(\varphi\left(\text{BN}(W_2 * \varphi(\text{BN}(W_1 * h^{(l)})))\right)\right)
$$

This progressively reduces spatial dimensions and increases semantic abstraction before final binary decision.

### Code-to-math mapping notes

- `nn.UpsamplingNearest2d(scale_factor=2)` and `nn.Upsample(..., scale_factor=2)` implement the $\text{Up}(\cdot)$ operator.
- `nn.MaxPool2d(2, 2)` is the $\text{Pool}(\cdot)$ downsampling in discriminator equations.
- `return self.conv_block(x) + self.shortcut(x)` is the explicit residual sum $F(x)+S(x) in the formula above.
- BatchNorm + activation pairs approximate affine-normalized nonlinear maps per channel at each stage.

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        # Linear projection maps latent vector z to a low-resolution feature tensor.
        self.initial_linear = nn.Linear(z_dim, 1024 * 4 * 4)
        
        self.net = nn.Sequential(
            GenBlock(1024, 512, is_final=False),
            GenBlock(512, 256, is_final=False),
            GenBlock(256, 128, is_final=False),
            GenBlock(128, 64, is_final=False),
            GenBlock(64, 64, is_final=True),
        )
        
        self.final_layer = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        if len(z.shape) > 2:
            z = z.view(z.size(0), -1)
            
        x = self.initial_linear(z)
        x = x.view(-1, 1024, 4, 4)
        x = self.net(x)
        return self.final_layer(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            DisBlock(3, 32),
            DisBlock(32, 64),
            DisBlock(64, 128),
        )
        
        # Output is a single logit used directly in BCEWithLogitsLoss.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 1),
        )

    def forward(self, x):
        x = self.net(x)
        return self.classifier(x)


class ResNetGenerator(nn.Module):
    def __init__(self, z_dim=100, base_channels=256):
        super().__init__()
        self.linear = nn.Linear(z_dim, 4 * 4 * base_channels)
        self.base_channels = base_channels

        self.blocks = nn.Sequential(
            ResGenBlock(base_channels, base_channels),
            ResGenBlock(base_channels, base_channels // 2),
            ResGenBlock(base_channels // 2, base_channels // 4),
            ResGenBlock(base_channels // 4, base_channels // 8),
            ResGenBlock(base_channels // 8, base_channels // 16),
        )
        
        self.final_layer = nn.Sequential(
            nn.BatchNorm2d(base_channels // 16),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels // 16, 3, 3, 1, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        if z.ndim > 2:
            z = z.view(z.size(0), -1)
            
        x = self.linear(z)
        x = x.view(-1, self.base_channels, 4, 4)
        x = self.blocks(x)
        return self.final_layer(x)


class GANModel(nn.Module):
    def __init__(self, z_dim=100, is_res=True):
        super().__init__()
        self.generator = Generator(z_dim) if not is_res else ResNetGenerator()
        self.discriminator = Discriminator()
        self.z_dim = z_dim
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, z):
        return self.generator(z)

    def compute_discriminator_loss(self, real_imgs, z):
        device = real_imgs.device
        
        # No gradients into generator during discriminator update.
        with torch.no_grad():
            fake_imgs = self.generator(z).detach()

        real_logits = self.discriminator(real_imgs)
        fake_logits = self.discriminator(fake_imgs)

        # One-sided label smoothing for real targets.
        real_labels = torch.full_like(real_logits, 0.9, device=device)
        real_loss = self.criterion(real_logits, real_labels)
        fake_labels = torch.zeros_like(fake_logits, device=device)
        fake_loss = self.criterion(fake_logits, fake_labels)

        d_loss = (real_loss + fake_loss) / 2
        return d_loss

    def compute_generator_loss(self, z):
        fake_imgs = self.generator(z)
        fake_logits = self.discriminator(fake_imgs)

        # Non-saturating GAN target: encourage D(G(z)) to predict real (label 1).
        target_labels = torch.ones_like(fake_logits, device=fake_logits.device)
        g_loss = self.criterion(fake_logits, target_labels)
        return g_loss, fake_imgs

## Model-Level Math: Latent Mapping, Logits, and Losses

### Latent-to-image mapping (`Generator`, `ResNetGenerator`)

A latent vector is sampled as:

$$
z \sim \mathcal{N}(0, I),\quad z\in\mathbb{R}^{d_z}
$$

then projected and reshaped to a seed feature map, then upsampled to image space:

$$
G_\theta: \mathbb{R}^{d_z} \to \mathbb{R}^{3\times H\times W}
$$

The current architecture upsamples from $4\times 4$ through five x2 stages, so output resolution becomes $128\times 128$ unless the block count is changed.

### Discriminator score

The discriminator outputs a scalar logit:

$$
s = D_\phi(x) \in \mathbb{R}
$$

Probability is implied by sigmoid:

$$
p(\text{real}\mid x)=\sigma(s)
$$

### Losses exactly as in `GANModel`

Discriminator loss in code (`compute_discriminator_loss`):

$$
\mathcal{L}_D = \frac{1}{2}\Big(\text{BCEWithLogits}(D(x_{real}), 0.9) + \text{BCEWithLogits}(D(G(z)), 0)\Big)
$$

- Real label smoothing uses $0.9$ (one-sided smoothing).
- Fake targets are $0$.

Generator loss (`compute_generator_loss`):

$$
\mathcal{L}_G = \text{BCEWithLogits}(D(G(z)), 1)
$$

This is the non-saturating generator objective, giving stronger gradients early in training.

### Code-to-math mapping notes

- `torch.full_like(real_logits, 0.9)` directly instantiates the smoothed real target in $\mathcal{L}_D$.
- `torch.zeros_like(fake_logits)` is the fake target term in $\mathcal{L}_D$.
- `torch.ones_like(fake_logits)` is the generator's adversarial target in $\mathcal{L}_G$.
- `with torch.no_grad(): fake_imgs = self.generator(z).detach()` enforces $\nabla_\theta \mathcal{L}_D = 0$ during discriminator updates.

In [ ]:
epochs = 1000
warmup_epochs = 10
dis_schedule = np.linspace(5, 1, num=warmup_epochs)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gan_model = GANModel().to(DEVICE)
opt_gen = torch.optim.Adam(gan_model.generator.parameters(), lr=1e-4)
opt_dis = torch.optim.Adam(gan_model.discriminator.parameters(), lr=1e-4)

train_dataset = CIFAR("/home/chaitanya-kohli/GAN-Him/tiny-imagenet/train/n07873807")
test_dataset = CIFAR("/home/chaitanya-kohli/GAN-Him/tiny-imagenet/val/n07873807")
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

gen_loss_list = []
dis_loss_list = []

for epoch in range(epochs):
    tot_gen_loss = 0.0
    tot_dis_loss = 0.0
    gan_model.train()
    tqdm_data = tqdm(train_loader, desc=f"Epoch-{epoch + 1}/{epochs}")
    
    # Early epochs bias more updates toward D, then anneal to balanced 1:1 updates.
    dis_steps = ceil(dis_schedule[epoch]) if epoch + 1 < warmup_epochs else 1
    i = 0

    for real_img in tqdm_data:
        i += 1
        real_img = real_img.to(DEVICE)
        bs = real_img.size(0)

        if i % dis_steps == 0:
            # z ~ N(0, I) for fake batch in discriminator objective.
            z_dis = torch.randn(bs, 100, device=DEVICE)
            opt_dis.zero_grad()
            dis_loss = gan_model.compute_discriminator_loss(real_img, z_dis)
            dis_loss.backward()
            opt_dis.step()
        else:
            dis_loss = torch.tensor(0.0, device=DEVICE)

        # Freeze D so generator step computes gradients only through G parameters.
        for param in gan_model.discriminator.parameters():
            param.requires_grad = False

        z = torch.randn(bs, 100, device=DEVICE)
        opt_gen.zero_grad()
        gen_loss, fake_img = gan_model.compute_generator_loss(z)
        gen_loss.backward()
        opt_gen.step()

        tot_gen_loss += gen_loss.detach().cpu().item()
        tot_dis_loss += dis_loss.detach().cpu().item()
        tqdm_data.set_postfix(
            {
                "GenLoss": gen_loss.detach().cpu().item(),
                "DisLoss": dis_loss.detach().cpu().item(),
                "DisSteps": dis_steps,
            }
        )

        # Restore D gradients for next discriminator step.
        for param in gan_model.discriminator.parameters():
            param.requires_grad = True

    gen_loss_list.append(tot_gen_loss / len(train_loader))
    dis_loss_list.append(tot_dis_loss * dis_steps / len(train_loader))

    print(f"Generator Loss: {gen_loss_list[-1]}\nDiscriminator Loss: {dis_loss_list[-1]}")

    if (epoch + 1) % 10 == 0:
        torch.save(
            gan_model.state_dict(),
            f"/home/chaitanya-kohli/GAN-Him/gan_outputs/weights/Experiment_pizza/pizza_gan_epoch_{epoch + 1}.pth",
        )
        gan_model.eval()
        with torch.no_grad():
            gan_image = gan_model(z)
            comparison = torch.cat([real_img[:8], gan_image[:8]], dim=0)
            grid = make_grid(comparison.cpu(), nrow=8, padding=2, normalize=True)
            plt.figure(figsize=(12, 4))
            plt.imshow(grid.permute(1, 2, 0))
            plt.axis("off")
            plt.title(f"Top: Original | Bottom: Generated Image (Epoch {epoch + 1})")
            plt.savefig(
                f"/home/chaitanya-kohli/GAN-Him/gan_outputs/plots/Experiment_pizza/pizza_Epoch-{epoch + 1}.png"
            )
            plt.show()
            plt.close()

plt.figure(figsize=(10, 5))
plt.title("Generator vs Discriminator Loss")
plt.plot(gen_loss_list, label="Generator")
plt.plot(dis_loss_list, label="Discriminator")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig("/home/chaitanya-kohli/GAN-Him/gan_outputs/plots/Experiment_pizza.png")
plt.legend()
plt.show()

## Training Loop Math: Alternating Game Optimization

The loop performs alternating stochastic optimization over mini-batches:

1. Update discriminator (sometimes multiple times early).
2. Freeze discriminator parameters.
3. Update generator once.
4. Unfreeze discriminator.

### Warmup discriminator schedule

`dis_schedule = np.linspace(5, 1, num=warmup_epochs)` means roughly:

$$
k_t \downarrow 5 \to 1
$$

where $k_t$ controls discriminator update frequency in early epochs. This can help avoid a too-weak discriminator at startup.

### Parameter updates (Adam)

For discriminator step:

$$
\phi \leftarrow \phi - \eta_D\,\nabla_\phi \mathcal{L}_D
$$

For generator step:

$$
\theta \leftarrow \theta - \eta_G\,\nabla_\theta \mathcal{L}_G
$$

where gradients are estimated from mini-batch expectations.

### Code-to-math mapping notes

- `z_dis = torch.randn(bs, 100, device=DEVICE)` is the explicit draw from $p_z = \mathcal{N}(0, I)$.
- `param.requires_grad = False` for discriminator parameters ensures only $\theta$ updates during generator backprop.
- `gen_loss.backward(); opt_gen.step()` implements one stochastic step for $\theta$.
- `dis_loss.backward(); opt_dis.step()` implements one stochastic step for $\phi$.
- Saving every 10 epochs materializes the trajectory of parameters $(\theta_t, \phi_t)$ over training time.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gan_model = GANModel().to(DEVICE)

# Load a pre-trained checkpoint before computing sample-quality metrics.
WEIGHT = torch.load("gan_outputs/mnist_gan.pth")
gan_model.load_state_dict(WEIGHT)

# Real-image reference distribution for FID/IS computation.
test_loader = DataLoader(
    MNISTCSV("/scratch/s25090/archive/mnist/mnist_test.csv"),
    batch_size=128,
    shuffle=False,
    )

In [ ]:
def get_evaluation_metrics(generator, dataloader, device, num_imgs=10000):
    """
    Calculates FID and IS for a GAN generator.

    Args:
        generator: The GAN generator model.
        dataloader: DataLoader for real images (needed for FID reference).
        device: 'cuda' or 'cpu'.
        num_imgs: Number of images to generate/use for calculation.
                  (Standard for papers is 50k, but 10k is faster for debugging).

    Returns:
        fid_score (float), is_score (float)
    """
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    inception = InceptionScore(normalize=True).to(device)

    generator.eval()
    print(f"--- Computing Metrics (Samples: {num_imgs}) ---")

    real_count = 0
    for batch, _ in tqdm(dataloader, desc="Processing Real Images"):
        batch = batch.to(device)
        remaining = num_imgs - real_count
        if remaining <= 0:
            break

        if batch.shape[0] > remaining:
            batch = batch[:remaining]

        # Metrics are computed on [0,1] images; training tensors may be in [-1,1].
        if batch.min() < 0:
            batch = (batch + 1) / 2

        fid.update((batch * 255).to(torch.uint8), real=True)
        real_count += batch.shape[0]

    fake_count = 0
    while fake_count < num_imgs:
        batch_size = min(dataloader.batch_size, num_imgs - fake_count)

        # Sample latent vectors and map to image manifold through G.
        z = torch.randn(batch_size, 100, device=device)
        with torch.no_grad():
            fake_imgs = generator(z)

        # Convert from tanh range [-1,1] to uint8 image domain for metric backbones.
        fake_imgs = (fake_imgs + 1) / 2
        fake_uint8 = (fake_imgs * 255).to(torch.uint8)

        fid.update(fake_uint8, real=False)
        inception.update(fake_uint8)
        fake_count += batch_size

    print("Finalizing calculations...")
    fid_score = fid.compute().item()
    is_score_mean, is_score_std = inception.compute()

    return fid_score, is_score_mean.item()


fid, is_score = get_evaluation_metrics(
    gan_model.generator,
    test_loader,
    DEVICE,
    num_imgs=2000,
    )
print(f"FID: {fid:.4f} | IS: {is_score:.4f}")

## Evaluation Math: FID and Inception Score

The metrics cell computes sample quality/diversity proxies.

### FID (Fr\'echet Inception Distance)

If Inception features for real and fake sets are modeled as Gaussians:

$$
(\mu_r, \Sigma_r),\; (\mu_g, \Sigma_g)
$$

then:

$$
\text{FID} = \|\mu_r-\mu_g\|_2^2 + \operatorname{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2}\right)
$$

Lower is better (closer to real distribution in feature space).

### Inception Score (IS)

Given class distribution $p(y\mid x)$ and marginal $p(y)$ over generated samples:

$$
\text{IS} = \exp\left(\mathbb{E}_{x}\left[D_{KL}(p(y\mid x)\,\|\,p(y))\right]\right)
$$

Higher is better (confident per-image predictions + diverse marginal classes).

### Why uint8 conversion appears in code

`torchmetrics` image metrics expect image-like ranges/types. Your code maps generated data from $[-1,1] \to [0,1]$ and then to $[0,255]$ as `uint8`, aligning with metric input assumptions.

### Code-to-math mapping notes

- `fid.update(real_batch, real=True)` accumulates moments for $(\mu_r, \Sigma_r)$.
- `fid.update(fake_batch, real=False)` accumulates moments for $(\mu_g, \Sigma_g)$.
- `inception.update(fake_batch)` accumulates $p(y|x)$ samples needed for IS.
- Truncating to `num_imgs` keeps both real and fake sample counts aligned for fair comparison.